# 04 · Oblique mounts, reflection tables, and g·b invisibility

The simplified geometry hard-wires the (−1,1,−1)@17 keV Al mount; oblique mode (v2.3.0) takes any cubic mount (`mount_x/y/z`) and any Laue-reachable reflection, and multi-reflection configs (v2.6.0) sweep several reflections in one run. The classic kinematical criterion says a dislocation vanishes when g·b = 0 ([Borgi 2025](../docs/references.md#borgi-2025); [Borgi 2024](../docs/references.md#borgi-2024) App. A) — below we watch it vanish.

In [ ]:
%matplotlib inline
import subprocess
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import PowerNorm

HERE = Path.cwd()
assert HERE.name == "examples", "Run this notebook from the examples/ folder"
IMG, OUT = HERE / "img", HERE / "out_04"
for d in (IMG, OUT):
    d.mkdir(exist_ok=True)

CONFIG = """
[reciprocal]
keV = 19.1
backend = "analytic"
beamstop = false

[geometry]
mode = "oblique"

[crystal]
lattice = "cubic"
a       = 4.0493e-10
mount_x = [1, 0, 0]
mount_y = [0, 1, 0]
mount_z = [0, 0, 1]
mode = "centered"

[crystal.centered]
b = [-1, 1, 0]
n = [1,  1, 1]
t = [-1, -1, 2]

[scan.phi]
value = 0.0
range = 1.25e-4
steps = 5

[io]
include_perfect_crystal = true

[[reflections]]
hkl = [1, 1, 1]

[[reflections]]
hkl = [2, 0, 0]
"""
cfg_file = OUT / "gb_invisibility.toml"
cfg_file.write_text(CONFIG, encoding="utf-8")
print(CONFIG)

## Which reflections can this mount even reach?

`dfxm-find-reflections` enumerates the Laue-reachable reflections for a mount and energy, with the Bragg angle and the two goniometer solutions for each.

In [ ]:
table = subprocess.run(
    [
        sys.executable,
        "-c",
        "from dfxm_geo.find_reflections_cmd import cli_main; cli_main()",
        "--config",
        str(cfg_file),
        "--hkl-max",
        "2",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(table.stdout)

## One run, two reflections, one invisible dislocation

The config's `[[reflections]]` blocks request g = (1,1,1) and g = (2,0,0) for the same b = ½[−110] dislocation: g·b = 0 for the first (invisible), g·b = −2 for the second (visible). The run writes one super-master with a per-reflection sub-tree each.

In [ ]:
from dfxm_geo.pipeline import SimulationConfig, run_simulation

cfg = SimulationConfig.from_toml(cfg_file)
result = run_simulation(cfg, OUT / "run")
print("reflections:", result["n_reflections"])

b = np.array([-1, 1, 0])
contrast = {}
images = {}
for i, g in enumerate(([1, 1, 1], [2, 0, 0]), start=1):  # reflection output dirs are 1-indexed
    master = OUT / "run" / f"reflection_{i:03d}" / "dfxm_geo.h5"
    with h5py.File(master, "r") as f:
        strained = f["/1.1/instrument/dfxm_sim_detector/data"][2]
        perfect = f["/2.1/instrument/dfxm_sim_detector/data"][2]
    contrast[tuple(g)] = float(np.std(strained - perfect) / max(float(np.mean(perfect)), 1e-10))
    images[tuple(g)] = strained

vmax = max(im.max() for im in images.values())
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, g in zip(axes, ([1, 1, 1], [2, 0, 0]), strict=False):
    gb = int(np.dot(g, b))
    ax.imshow(images[tuple(g)], cmap="magma", norm=PowerNorm(0.45, vmin=0.0, vmax=vmax))
    ax.set_title(f"g={g}  g·b={gb}  contrast={contrast[tuple(g)]:.3f}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(IMG / "04_reflections_preview.png", dpi=110, bbox_inches="tight")

ratio = contrast[(2, 0, 0)] / contrast[(1, 1, 1)]
print(f"contrast ratio (visible/invisible) = {ratio:.2f}")
assert ratio > 3.0, "g.b invisibility broke - expected >3x"

The g·b = 0 panel keeps only faint residual contrast, while g·b = −2 shows the full dislocation — this centre-frame (rocking-peak) metric gives ≈4.6×; the test suite's off-peak gate asserts the same physics at >3×. This dislocation is pure edge (t = n×b), so that faint residue is the edge-character residual — the g·b = 0 criterion is exact only for screw dislocations. The same `[[reflections]]` mechanism drives whole sweep campaigns, and `exclude_invisibility` uses g·b to skip blind configurations in identification. Next: [05 · Identification at scale](05_identification_at_scale.ipynb).